In [1]:
# ── Cell 1: Imports & Setup ──
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy.special import expit

# Ensure data folder exists
os.makedirs("../data", exist_ok=True)

# Reproducibility
np.random.seed(42)
n = 1000


In [2]:
# ── Cell 2: Generate Raw ER Features ──
df = pd.DataFrame({
    "patient_id": np.arange(1, n+1),
    "age":         np.random.randint(18, 90, n),
    "gender":      np.random.choice(["Male","Female"], n),
    "arrival_time":[
        datetime(2024,5,1) + timedelta(minutes=int(x))
        for x in np.random.exponential(scale=100, size=n)
    ],
    "triage_score": np.random.choice([1,2,3,4,5], n),
    "heart_rate":   np.random.normal(85, 15, n).astype(int),
    "systolic_bp":  np.random.normal(130,20,n).astype(int),
    "diastolic_bp": np.random.normal(85,10,n).astype(int),
})
df["arrival_to_triage_mins"] = np.random.randint(2,45,n)
df["visit_reason"] = np.random.choice(
    ["Chest Pain","Fever","Injury","Shortness of Breath","Dizziness"], n
)
print("Raw sample:")
print(df.head(3))


Raw sample:
   patient_id  age  gender        arrival_time  triage_score  heart_rate  \
0           1   69    Male 2024-05-01 01:29:00             2          89   
1           2   32    Male 2024-05-01 01:26:00             4          98   
2           3   89  Female 2024-05-01 01:48:00             4          84   

   systolic_bp  diastolic_bp  arrival_to_triage_mins         visit_reason  
0          126            78                      29  Shortness of Breath  
1          121            78                       3               Injury  
2          120            76                      11               Injury  


In [ ]:
# ── Cell 3: Embed a Balanced Signal into 'admitted' & 'readmitted' ──
from scipy.special import expit

# 1) Compute bp_delta
df['bp_delta'] = df['systolic_bp'] - df['diastolic_bp']

# 2) Raw linear combination (use whichever coefficients you like)-- 
coeffs = {
    'triage_score':     2.0,
    'heart_rate':       0.1,
    'bp_delta':         0.05,
    'arrival_to_triage_mins': -0.1,
    'age':              0.03
}
intercept = -4.5
# similar to logistic regression-> it takes each feature, multiplies it by a weight (coefficient), adds them all together plus a constant (intercept)
linear_raw = (
      coeffs['triage_score']     * df['triage_score']
    + coeffs['heart_rate']       * df['heart_rate']
    + coeffs['bp_delta']         * df['bp_delta']
    + coeffs['arrival_to_triage_mins'] * df['arrival_to_triage_mins']
    + coeffs['age']              * df['age']
    + intercept
)

# 3) Center the linear term around its median → ~50/50 classes
linear_centered = linear_raw - np.median(linear_raw)

# 4) Convert to probabilities and draw labels
df['admit_prob'] = expit(linear_centered)
df['admitted']  = (np.random.rand(len(df)) < df['admit_prob']).astype(int)

# 5) Readmission tied to admit_prob & age (you can similarly center here)--log odds
read_raw = -2 + 0.7*df['admit_prob'] + 0.04*df['age']
read_centered = read_raw - np.median(read_raw)
df['readmit_prob']  = expit(read_centered)
df['readmitted']    = (np.random.rand(len(df)) < df['readmit_prob']).astype(int)

# 6) Sanity-check distributions
print("Admit distribution:\n", df['admitted'].value_counts(normalize=True))
print("Readmit distribution:\n", df['readmitted'].value_counts(normalize=True))

# 7) Clean up helpers
df.drop(columns=['admit_prob','readmit_prob'], inplace=True)


Admit distribution:
 admitted
1    0.508
0    0.492
Name: proportion, dtype: float64
Readmit distribution:
 readmitted
1    0.506
0    0.494
Name: proportion, dtype: float64


In [4]:
# ── Cell 4: Save to CSV ──
out_path = "../data/synthetic_er_data.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df)} records with true signal to {out_path}")


Wrote 1000 records with true signal to ../data/synthetic_er_data.csv
